In [ ]:
import os
import csv
import time
import datetime as dt
import pandas as pd
import praw
import pandas as pd
import networkx as nx
from collections import defaultdict
import itertools
import matplotlib.pyplot as plt
import numpy as np


# ==================================================
# REDDIT API CONFIGURATION
# Insert your credentials below before running
# ==================================================

os.environ["REDDIT_CLIENT_ID"] = "INSERT_CLIENT_ID"
os.environ["REDDIT_CLIENT_SECRET"] = "INSERT_CLIENT_SECRET"
os.environ["REDDIT_USER_AGENT"] = "sna-project "

# ==================================================
# CONFIGURATION
# ==================================================

# Crawling time window
AFTER  = "2026-03-15"   # inclusive
BEFORE = "2026-03-17"   # exclusive

# Political home communities
LEFT_HOMES = [
    "Socialism",
    "LateStageCapitalism",
    "Anarchism",
    "SocialDemocracy",
    "progressive",
    "AskALiberal"
]

RIGHT_HOMES = [
    "Conservative",
    "Libertarian",
    "AskConservatives",
    "Capitalism",
    "ConservativeMemes"
]

# Output file
BASE_DIR = os.getcwd()
OUT_CSV = os.path.join(BASE_DIR, "seed_homes.csv")

# Crawling parameters
SLEEP = 0.30
MAX_ROWS_PER_SIDE = 200_000
PROGRESS_EVERY = 5000


# ==================================================
# UTILITY FUNCTIONS
# ==================================================

def to_epoch(date_str: str) -> int:
    """Convert YYYY-MM-DD date string to Unix timestamp."""
    return int(
        dt.datetime.strptime(date_str, "%Y-%m-%d")
        .replace(tzinfo=dt.timezone.utc)
        .timestamp()
    )


def fmt_eta(seconds: float) -> str:
    """Format seconds into readable ETA."""
    seconds = max(0, int(seconds))
    h, r = divmod(seconds, 3600)
    m, s = divmod(r, 60)

    if h:
        return f"{h}h {m:02d}m {s:02d}s"
    return f"{m}m {s:02d}s"


def reddit_client():
    """Create Reddit API client using environment variables."""
    cid = os.environ.get("REDDIT_CLIENT_ID")
    csc = os.environ.get("REDDIT_CLIENT_SECRET")
    ua  = os.environ.get("REDDIT_USER_AGENT", "sna-project/1.0")

    if not cid or not csc:
        raise RuntimeError(
            "Missing Reddit API credentials. "
            "Insert REDDIT_CLIENT_ID and REDDIT_CLIENT_SECRET."
        )

    return praw.Reddit(
        client_id=cid,
        client_secret=csc,
        user_agent=ua,
        check_for_async=False
    )


# ==================================================
# CRAWL FUNCTION (for one political side)
# ==================================================

def crawl_side(reddit, subs, side, after_epoch, before_epoch, writer, max_rows):

    rows = 0
    users_seen = set()
    t0 = time.time()

    for sub in subs:

        for submission in reddit.subreddit(sub).new(limit=None):

            ts = int(getattr(submission, "created_utc", 0) or 0)

            if ts >= before_epoch:
                continue

            if ts < after_epoch:
                break

            # Download comments from the thread
            submission.comments.replace_more(limit=0)

            for comment in submission.comments.list():

                author = str(getattr(comment, "author", "") or "")

                if not author or author == "AutoModerator":
                    continue

                comment_time = int(getattr(comment, "created_utc", 0) or 0)

                if not (after_epoch <= comment_time < before_epoch):
                    continue

                thread_id = f"t3_{submission.id}"

                writer.writerow({
                    "author": author,
                    "subreddit": sub,
                    "side": side,
                    "created_utc": comment_time,
                    "link_id": thread_id
                })

                rows += 1
                users_seen.add(author)

                # Progress monitor
                if rows % PROGRESS_EVERY == 0:

                    elapsed = time.time() - t0
                    speed = rows / max(1e-9, elapsed)
                    remaining = max_rows - rows
                    eta = remaining / max(1e-9, speed)

                    print(
                        f"[{side}] {rows:,}/{max_rows:,} rows | "
                        f"users≈{len(users_seen):,} | "
                        f"elapsed={fmt_eta(elapsed)} | ETA≈{fmt_eta(eta)} | sub={sub}"
                    )

                if rows >= max_rows:

                    elapsed = time.time() - t0

                    print(
                        f"[{side}] STOP: reached limit {max_rows:,} rows | "
                        f"users≈{len(users_seen):,} | time={fmt_eta(elapsed)}"
                    )

                    return rows, users_seen

            time.sleep(SLEEP)

    elapsed = time.time() - t0

    print(
        f"[{side}] DONE: {rows:,} rows | users≈{len(users_seen):,} | "
        f"time={fmt_eta(elapsed)}"
    )

    return rows, users_seen


# ==================================================
# RUN CRAWLING
# ==================================================

os.makedirs(BASE_DIR, exist_ok=True)

after, before = to_epoch(AFTER), to_epoch(BEFORE)

reddit = reddit_client()

total_left = total_right = 0
left_users = set()
right_users = set()


with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:

    writer = csv.DictWriter(
        f,
        fieldnames=["author", "subreddit", "side", "created_utc", "link_id"]
    )

    writer.writeheader()

    print(f"▶️  Crawl LEFT | window {AFTER} → {BEFORE} | max_rows={MAX_ROWS_PER_SIDE:,}")

    rows_left, users_left = crawl_side(
        reddit, LEFT_HOMES, "left", after, before, writer, MAX_ROWS_PER_SIDE
    )

    total_left += rows_left
    left_users |= users_left


    print(f"\n▶️  Crawl RIGHT | window {AFTER} → {BEFORE} | max_rows={MAX_ROWS_PER_SIDE:,}")

    rows_right, users_right = crawl_side(
        reddit, RIGHT_HOMES, "right", after, before, writer, MAX_ROWS_PER_SIDE
    )

    total_right += rows_right
    right_users |= users_right


print("\n✅ COMPLETED")
print(f"Left  : rows={total_left:,} | users≈{len(left_users):,}")
print(f"Right : rows={total_right:,} | users≈{len(right_users):,}")
print(f"CSV   → {OUT_CSV}")


# ==================================================
# QUICK SUMMARY
# ==================================================

df_seed = pd.read_csv(OUT_CSV)

print("\n--- SUMMARY ---")
print(df_seed["side"].value_counts())

print("Unique authors LEFT :", df_seed.loc[df_seed["side"] == "left", "author"].nunique())
print("Unique authors RIGHT:", df_seed.loc[df_seed["side"] == "right", "author"].nunique())

In [ ]:
# ==================================================
# ARENA SUBREDDITS (mixed political discussion)
# ==================================================

ARENAS = [
    "worldnews",
    "news",
    "geopolitics",
    "NeutralPolitics",
    "PoliticalDiscussion",
    "IsraelPalestine",
    "MiddleEast",
    "politics"
]

# Input seed users (from home communities crawl)
SEED_FILE = "seed_homes.csv"

# Output file
OUT_CSV = os.path.join(BASE_DIR, "arena_activity.csv")

# Crawling parameters
MAX_POSTS_PER_SUB = 100
MAX_COMMENTS_PER_POST = 300
SLEEP = 0.3
PROGRESS_EVERY = 2000


# ==================================================
# CRAWL MIDDLE-GROUND SUBREDDITS
# ==================================================

def crawl_middle_ground(arenas, seed_users, after, before, out_csv):

    reddit = reddit_client()

    after_epoch = to_epoch(after)
    before_epoch = to_epoch(before)

    os.makedirs(os.path.dirname(out_csv) or ".", exist_ok=True)

    total_rows = 0
    seen_users = set()
    started = time.time()

    with open(out_csv, "w", newline="", encoding="utf-8") as f:

        writer = csv.DictWriter(
            f,
            fieldnames=["author", "subreddit", "link_id", "created_utc"]
        )
        writer.writeheader()

        for sub in arenas:

            posts_seen = 0
            print(f"\n=== Crawling arena: r/{sub} ===")

            for submission in reddit.subreddit(sub).new(limit=None):

                ts = int(getattr(submission, "created_utc", 0) or 0)

                if ts >= before_epoch:
                    continue

                if ts < after_epoch:
                    break

                posts_seen += 1

                if posts_seen > MAX_POSTS_PER_SUB:
                    break

                submission.comments.replace_more(limit=0)

                comment_counter = 0

                for comment in submission.comments.list():

                    author = str(getattr(comment, "author", "") or "")

                    if author not in seed_users:
                        continue

                    comment_time = int(getattr(comment, "created_utc", 0) or 0)

                    if not (after_epoch <= comment_time < before_epoch):
                        continue

                    writer.writerow({
                        "author": author,
                        "subreddit": sub,
                        "link_id": f"t3_{submission.id}",
                        "created_utc": comment_time
                    })

                    total_rows += 1
                    seen_users.add(author)
                    comment_counter += 1

                    # Progress monitor
                    if total_rows % PROGRESS_EVERY == 0:

                        elapsed = time.time() - started
                        rate = total_rows / max(1, elapsed)

                        estimated_total = (
                            MAX_POSTS_PER_SUB
                            * len(arenas)
                            * MAX_COMMENTS_PER_POST
                        )

                        eta = (estimated_total - total_rows) / max(1, rate)

                        print(
                            f"{total_rows:,} rows | "
                            f"users≈{len(seen_users):,} | "
                            f"elapsed={fmt_eta(elapsed)} | "
                            f"ETA≈{fmt_eta(eta)} | "
                            f"sub={sub}"
                        )

                    if comment_counter >= MAX_COMMENTS_PER_POST:
                        break

                time.sleep(SLEEP)

    elapsed = time.time() - started

    print(
        f"\n✅ Arena crawl completed: {total_rows:,} rows | "
        f"users≈{len(seen_users):,} | "
        f"time={fmt_eta(elapsed)}"
    )

    return out_csv


# ==================================================
# RUN ARENA CRAWL
# ==================================================

df_seed = pd.read_csv(SEED_FILE)
seed_users = set(df_seed["author"])

print(f"Seed users loaded: {len(seed_users):,}")

crawl_middle_ground(
    ARENAS,
    seed_users,
    AFTER,
    BEFORE,
    OUT_CSV
)


# ==================================================
# QUICK REPORT
# ==================================================

df_arena = pd.read_csv(OUT_CSV)

print("\n--- SUMMARY ---")
print("Total comments:", len(df_arena))
print("Unique users :", df_arena["author"].nunique())
print("Subreddits   :", df_arena["subreddit"].nunique())

print("\nTop 10 subreddits:")
print(df_arena["subreddit"].value_counts().head(10))

In [ ]:
SEED_CSV          = "seed_homes.csv"                 # deve contenere: author, subreddit, side, created_utc, link_id
ARENA_CSV_RAW     = "arena_activity.csv"             # non etichettato o già etichettato, va bene
ARENA_LABELED_OUT = "arena_activity_labeled.csv"
COMBINED_OUT      = "combined_seed_arena.csv"

# --- 1) mapping author -> side from seed ---
seed = pd.read_csv(SEED_CSV)
seed["author"] = seed["author"].astype(str).str.strip()
seed_side_map = (seed[["author","side"]]
                 .dropna()
                 .drop_duplicates(subset=["author","side"]))
# if two or more side , keep more frequent 
if seed_side_map.duplicated("author").any():
    counts = (seed.groupby(["author","side"]).size()
                    .rename("n").reset_index())
    seed_side_map = (counts.sort_values(["author","n"], ascending=[True,False])
                           .drop_duplicates("author")[["author","side"]])

# --- 2) label arena 
arena = pd.read_csv(ARENA_CSV_RAW)
arena["author"] = arena["author"].astype(str).str.strip()
arena_labeled = arena.merge(seed_side_map, on="author", how="left")
arena_labeled.to_csv(ARENA_LABELED_OUT, index=False)

# --- 3) add 'source'  allign by name, ---
seed_with_source  = seed.copy()
arena_with_source = arena_labeled.copy()
seed_with_source["source"]  = "home"
arena_with_source["source"] = "arena"

cols = ["author","subreddit","link_id","created_utc","side","source"]


# --- 4) concat and save---
combined = pd.concat([seed_with_source,arena_with_source], ignore_index=True)
combined.to_csv(COMBINED_OUT, index=False)

# --- 5) quick check  ---
print("✔️ Salvati:")
print("  -", ARENA_LABELED_OUT)
print("  -", COMBINED_OUT)

print("\n— Sanity check —")
print("Valori 'side' unici (prime 10):", combined["side"].dropna().astype(str).unique()[:10])
print("Esempi 'link_id' (prime 10):", combined["link_id"].dropna().astype(str).unique()[:10])
print("\nCounts per source:")
print(combined["source"].value_counts(dropna=False).to_string())
print("\nUtenti unici per source:")
print(combined.groupby("source")["author"].nunique())

In [ ]:
# === CONFIG ===
COMBINED_CSV = "combined_seed_arena.csv"
OUT_GEXF     = "G_full_co_thread.gexf"
OUT_GRAPHML  = "G_full_co_thread.graphml"
MIN_SHARED_THREADS = 1  

# === 1) load ===
df = pd.read_csv(COMBINED_CSV)
print(f"Totale righe: {len(df):,} | utenti unici: {df['author'].nunique():,}")

# keep raws with author, link_id and side defined
df = df.dropna(subset=["author","link_id","side"])
df["author"] = df["author"].astype(str)
df["link_id"] = df["link_id"].astype(str)
df["side"] = df["side"].astype(str)

# === 2) build edges co-thread ===
print("Costruzione coppie (u,v) per thread...")
pair_w = defaultdict(int)
for lid, g in df.groupby("link_id"):
    users = sorted(g["author"].unique())
    for u, v in itertools.combinations(users, 2):
        pair_w[(u, v)] += 1


if MIN_SHARED_THREADS > 1:
    pair_w = {e:w for e,w in pair_w.items() if w >= MIN_SHARED_THREADS}

print(f"Archi trovati: {len(pair_w):,}")

# === 3) builf graph ===
G = nx.Graph()
# add nodes
G.add_nodes_from(df["author"].unique())

# attributes
side_map = dict(zip(df["author"], df["side"]))
nx.set_node_attributes(G, {u: {"side": side_map.get(u)} for u in G.nodes})

# add edge 
for (u,v),w in pair_w.items():
    G.add_edge(u, v, weight=int(w))

print(f"Grafo creato: nodi={G.number_of_nodes():,}, archi={G.number_of_edges():,}")

# === 4) Save ===
nx.write_gexf(G, OUT_GEXF)
nx.write_graphml(G, OUT_GRAPHML)
print(f"Salvato: {OUT_GEXF} e {OUT_GRAPHML}")

# === 5) Quick Report  ===
if G.number_of_nodes() > 0 and G.number_of_edges() > 0:
    try:
        assort = nx.attribute_assortativity_coefficient(G, "side")
        print(f"Assortatività (side): {assort:.3f}")
    except Exception as e:
        print("Assortatività non calcolabile:", e)

    dens = nx.density(G)
    print(f"Densità del grafo: {dens:.5f}")

    try:
        cc = nx.average_clustering(G, weight="weight")
        print(f"Clustering medio: {cc:.3f}")
    except Exception as e:
        print("Clustering non calcolabile:", e)

In [ ]:

# ========================
# 1. GRAFO REALE
# ========================

G = nx.read_gexf("G_full_co_thread.gexf")

deg_real = [d for n, d in G.degree()]
N = G.number_of_nodes()
M = G.number_of_edges()

print(f"Real graph: N={N:,}, M={M:,}")

# ========================
# 2.BARABÁSI–ALBERT (BA)
# ========================
# Per un BA, m ≈ M/N
m = max(1, int(M / N))

print(f"Generating BA model with N={N}, m={m}")

G_ba = nx.barabasi_albert_graph(N, m)
deg_ba = [d for n, d in G_ba.degree()]

# ========================
# 3. DEGREE DISTRIBUTION
# ========================
def degree_distribution(deg_list):
    values, counts = np.unique(deg_list, return_counts=True)
    probs = counts / counts.sum()
    return values, probs

k_real, p_real = degree_distribution(deg_real)
k_ba, p_ba   = degree_distribution(deg_ba)

# ========================
# 4. PLOT 
# ========================
plt.figure(figsize=(8,6))
plt.scatter(k_real, p_real, s=25, alpha=0.7, label="Real Network")
plt.scatter(k_ba, p_ba, s=25, alpha=0.7, label="BA Model")

plt.xscale("log")
plt.yscale("log")
plt.xlabel("Degree k (log scale)")
plt.ylabel("P(k) (log scale)")
plt.title("Degree Distribution: Real Network vs Barabási–Albert (BA)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()